# Notebook 02 — Data Cleaning & ETL Pipeline

**Objective:** Build a fully documented, reproducible Python ETL pipeline that transforms the raw VC investment dataset into a clean, analysis-ready file. Every transformation is logged with a reason.

In [ ]:
import pandas as pd

import numpy as np

import warnings

warnings.filterwarnings('ignore')

RAW_PATH       = '/content/investments_VC (1).csv'

CLEAN_PATH     = 'startups_cleaned.csv'

etl_log = []

def log_step(step: int, desc: str, before: int, after: int, detail: str = ""):

    """Append a row to the audit log and print a summary line."""

    dropped = before - after

    pct_kept = after / before * 100 if before > 0 else 0

    etl_log.append({

        "Step"       : step,

        "Description": desc,

        "Before"     : before,

        "After"      : after,

        "Dropped"    : dropped,

        "% Kept"     : round(pct_kept, 1),

        "Detail"     : detail,

    })

    flag = "⚠️" if dropped > 500 else "✅"

    print(f"{flag} [Step {step:02d}] {desc}")

    print(f"          {before:,} → {after:,} rows  |  dropped {dropped:,}  |  {pct_kept:.1f}% retained")

print("Configuration complete.")

In [ ]:
df = pd.read_csv(RAW_PATH, encoding='latin-1', low_memory=False)

initial_rows = len(df)

print(f"Raw dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

print(f"\nColumn names (raw):")

print(df.columns.tolist())

print(f"\nData types:")

print(df.dtypes)

print(f"\nFirst 3 rows:")

df.head(3)

In [ ]:

print("=== INITIAL MISSING VALUE AUDIT ===")

missing = df.isnull().sum()

missing_pct = (missing / len(df) * 100).round(1)

audit = pd.DataFrame({'Missing Count': missing, '% Missing': missing_pct})

print(audit[audit['Missing Count'] > 0].to_string())

In [ ]:
before = len(df)

original_cols = df.columns.tolist()

df.columns = df.columns.str.strip()

renamed = [(o, n) for o, n in zip(original_cols, df.columns.tolist()) if o != n]

log_step(1, 'Strip whitespace from column names', before, len(df),

         f'Renamed columns: {renamed}')

print(f"\nColumns renamed: {renamed}")

In [ ]:
before = len(df)

def clean_funding(val):

    """Convert a funding string like '1,750,000' or ' - ' to float. Returns NaN for unknowns."""

    if pd.isna(val):

        return np.nan

    val_str = str(val).strip().replace(',', '').replace(' ', '')

    if val_str in ['-', '', 'nan', 'None', 'N/A']:

        return np.nan

    try:

        return float(val_str)

    except ValueError:

        return np.nan

df['funding_total_usd'] = df['funding_total_usd'].apply(clean_funding)

log_step(2, 'Convert funding_total_usd to numeric', before, len(df),

         'Stripped commas, dashes, spaces; converted to float64; dashes → NaN')

print(f"\nFunding stats after conversion:")

print(df['funding_total_usd'].describe().apply(lambda x: f'{x:,.0f}'))

In [ ]:
before = len(df)

date_cols = ['founded_at', 'first_funding_at', 'last_funding_at']

for col in date_cols:

    df[col] = pd.to_datetime(df[col], errors='coerce')

log_step(3, 'Parse date columns to datetime64', before, len(df),

         f'Columns converted: {date_cols}. Unparseable strings coerced to NaT.')

print("\nDate column null counts after parsing:")

print(df[date_cols].isnull().sum())

In [ ]:
before = len(df)

mask_a = (

    df['first_funding_at'].isna() |

    df['founded_at'].isna() |

    (df['first_funding_at'] >= df['founded_at'])

)

df = df[mask_a]

mask_b = (

    df['last_funding_at'].isna() |

    df['first_funding_at'].isna() |

    (df['last_funding_at'] >= df['first_funding_at'])

)

df = df[mask_b]

log_step(4, 'Enforce date logical ordering', before, len(df),

         'Removed rows where first_funding < founded_at, or last_funding < first_funding')

In [ ]:
before = len(df)

str_cols = ['name', 'market', 'status', 'country_code', 'state_code',

            'region', 'city', 'category_list', 'permalink', 'homepage_url']

null_markers = {'nan', 'none', 'n/a', 'na', '', 'null', '-'}

for col in str_cols:

    if col in df.columns:

        df[col] = df[col].astype(str).str.strip()

        df[col] = df[col].apply(lambda x: np.nan if x.lower() in null_markers else x)

log_step(5, 'Standardise string columns and unify null markers', before, len(df),

         f'Cleaned {len(str_cols)} columns. Null markers unified to NaN.')

print("\nPost-step sample — unique status values:")

print(df['status'].value_counts(dropna=False))

In [ ]:
before = len(df)

print("Raw status distribution (before standardisation):")

print(df['status'].value_counts(dropna=False).to_string())

df['status'] = df['status'].str.lower().str.strip()

status_map = {

    'operating': 'operating',

    'closed'   : 'closed',

    'acquired' : 'acquired',

    'ipo'      : 'ipo',

}

df['status'] = df['status'].map(status_map)

log_step(6, 'Map status to 4 canonical categories', before, len(df),

         'Values not in {operating, closed, acquired, ipo} set to NaN for next step')

print("\nCleaned status distribution:")

print(df['status'].value_counts(dropna=False).to_string())

In [ ]:
before = len(df)

df = df.dropna(subset=['status'])

log_step(7, 'Drop rows with missing status (target variable)', before, len(df),

         'Cannot impute or infer the outcome label for the "Why Startups Fail" analysis')

In [ ]:
before = len(df)

dedup_key = ['name', 'country_code', 'founded_year']

df = df.drop_duplicates(subset=dedup_key, keep='first')

log_step(8, f'Remove duplicates on {dedup_key}', before, len(df),

         'Kept first occurrence. Duplicates indicate same startup entered multiple times.')

In [ ]:
before = len(df)

round_cols = [

    'seed', 'venture', 'equity_crowdfunding', 'undisclosed',

    'convertible_note', 'debt_financing', 'angel', 'grant',

    'private_equity', 'post_ipo_equity', 'post_ipo_debt',

    'secondary_market', 'product_crowdfunding',

    'round_A', 'round_B', 'round_C', 'round_D',

    'round_E', 'round_F', 'round_G', 'round_H'

]

round_cols_present = [c for c in round_cols if c in df.columns]

for col in round_cols_present:

    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

log_step(9, 'Convert funding round columns to int (NaN → 0)', before, len(df),

         f'{len(round_cols_present)} round columns standardised. NaN = no funding received = 0.')

print(f"\nRound columns processed: {round_cols_present}")

In [ ]:
before = len(df)

Q1  = df['funding_total_usd'].quantile(0.25)

Q3  = df['funding_total_usd'].quantile(0.75)

IQR = Q3 - Q1

upper_fence = Q3 + 3 * IQR

outliers_above = (df['funding_total_usd'] > upper_fence).sum()

df['funding_total_usd'] = df['funding_total_usd'].clip(upper=upper_fence)

log_step(10, 'Cap funding outliers at Q3 + 3×IQR', before, len(df),

         f'Q1={Q1:,.0f}, Q3={Q3:,.0f}, IQR={IQR:,.0f}. '

         f'Upper fence={upper_fence:,.0f}. Capped {outliers_above} rows (not dropped).')

print(f"\nOutlier cap applied: ${upper_fence:,.0f}")

print(f"Values capped: {outliers_above}")

print(f"\nPost-cap funding stats:")

print(df['funding_total_usd'].describe().apply(lambda x: f'{x:,.0f}'))

In [ ]:
before = len(df)

YEAR_MIN, YEAR_MAX = 1990, 2014

df = df[

    df['founded_year'].isna() |

    ((df['founded_year'] >= YEAR_MIN) & (df['founded_year'] <= YEAR_MAX))

]

log_step(11, f'Filter to founded year {YEAR_MIN}–{YEAR_MAX}', before, len(df),

         'Pre-1990 data is sparse and structurally different. Max year in dataset is 2014.')

print(f"\nFounded year distribution after filter:")

print(df['founded_year'].value_counts().sort_index().to_string())

In [ ]:
before = len(df)

df['days_to_first_funding'] = (df['first_funding_at'] - df['founded_at']).dt.days

neg_mask = df['days_to_first_funding'] < 0

print(f"Rows with negative days_to_first_funding: {neg_mask.sum():,}")

df = df[df['days_to_first_funding'].isna() | (df['days_to_first_funding'] >= 0)]

log_step(12, 'Remove rows where funding predates founding', before, len(df),

         'days_to_first_funding < 0 means first_funding_at < founded_at — data entry error')

In [ ]:
before = len(df)

df = df.dropna(subset=['funding_total_usd'])

log_step(13, 'Drop rows missing funding_total_usd', before, len(df),

         'Core financial metric — cannot be accurately imputed for analysis')

In [ ]:
before = len(df)

df = df.dropna(subset=['founded_year'])

log_step(14, 'Drop rows missing founded_year', before, len(df),

         'Required for temporal/cohort analysis. Cannot be reliably imputed.')

In [ ]:
before = len(df)

df['funding_duration_days'] = (df['last_funding_at'] - df['first_funding_at']).dt.days

df['avg_funding_per_round'] = np.where(

    df['funding_rounds'] > 0,

    df['funding_total_usd'] / df['funding_rounds'],

    0

)

df['is_usa'] = (df['country_code'] == 'USA').astype(int)

df['primary_category'] = (

    df['category_list']

    .astype(str)

    .str.split('|')

    .apply(lambda parts: next((p.strip() for p in parts if p.strip() not in ['', 'nan', 'None']), np.nan))

)

df['is_closed'] = (df['status'] == 'closed').astype(int)

series_cols = [c for c in ['round_A','round_B','round_C','round_D','round_E','round_F','round_G','round_H'] if c in df.columns]

df['reached_series_a'] = (df[series_cols].sum(axis=1) > 0).astype(int)

df['founding_decade'] = (df['founded_year'] // 10 * 10).astype('Int64')

def funding_tier(usd):

    if pd.isna(usd) or usd == 0: return 'Unknown'

    if usd < 100_000:            return '<$100K'

    if usd < 1_000_000:          return '$100K–$1M'

    if usd < 10_000_000:         return '$1M–$10M'

    if usd < 50_000_000:         return '$10M–$50M'

    return '$50M+'

df['funding_tier'] = df['funding_total_usd'].apply(funding_tier)

if 'seed' in df.columns:

    df['has_seed'] = (df['seed'] > 0).astype(int)

log_step(15, 'Feature engineering', before, len(df),

         'Added: days_to_first_funding, funding_duration_days, avg_funding_per_round, '

         'is_usa, primary_category, is_closed, reached_series_a, founding_decade, '

         'funding_tier, has_seed')

print("\nNew features summary:")

new_feat_cols = ['days_to_first_funding','funding_duration_days','avg_funding_per_round',

                 'is_closed','reached_series_a','has_seed']

print(df[[c for c in new_feat_cols if c in df.columns]].describe().round(2).to_string())

In [ ]:
print("=" * 60)

print("FINAL DATASET QUALITY REPORT")

print("=" * 60)

print(f"\nShape:   {df.shape[0]:,} rows × {df.shape[1]} columns")

print(f"Memory:  {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

print("\n── Status distribution (target variable) ──")

status_dist = df['status'].value_counts()

for status, count in status_dist.items():

    pct = count / len(df) * 100

    print(f"  {status:12s}: {count:>6,}  ({pct:.1f}%)")

print("\n── Failure rate (closed) by funding tier ──")

print(df.groupby('funding_tier')['is_closed'].mean().sort_values(ascending=False).round(3).to_string())

print("\n── Remaining missing values in key columns ──")

key_cols = ['status','funding_total_usd','country_code','founded_year',

            'funding_rounds','is_closed','primary_category']

missing_final = df[key_cols].isnull().sum()

print(missing_final.to_string())

print("\n── Top 10 markets (by startup count) ──")

if 'market' in df.columns:

    print(df['market'].value_counts().head(10).to_string())

In [ ]:
etl_df = pd.DataFrame(etl_log)

print("=" * 80)

print("ETL PIPELINE AUDIT LOG")

print("=" * 80)

print(etl_df[['Step','Description','Before','After','Dropped','% Kept']].to_string(index=False))

print()

print(f"  Initial rows : {initial_rows:,}")

print(f"  Final rows   : {len(df):,}")

print(f"  Total dropped: {initial_rows - len(df):,}  ({(initial_rows - len(df)) / initial_rows * 100:.1f}% of raw data)")

print(f"  Rows retained: {len(df) / initial_rows * 100:.1f}%")

In [ ]:

cols_to_drop = ['permalink', 'homepage_url']

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

df.to_csv(CLEAN_PATH, index=False)

print(f"✅ Cleaned dataset saved to: {CLEAN_PATH}")

print(f"   {len(df):,} rows × {df.shape[1]} columns")

try:

    from google.colab import files

    files.download(CLEAN_PATH)

    print("📥 Download triggered (Colab).")

except ImportError:

    print("ℹ️  Not running on Colab — file saved locally.")

print(f"\nFinal columns ({df.shape[1]}):")

print(df.columns.tolist())